Parameter optimization
================

This notebook shows how both hyperparameters and data source weights can be optimized. This code merely serves as a demonstration of the process. For the full optimization code see [run optimization](../optimization/run_optimization.py) and [combine optimization folds](../optimization/combine_optimization_folds.ipynb). 

In [ ]:
from nichenetpy.utils import (
    read_csv_cols
)
from nichenetpy.parameter_optimization import (
    construct_and_evaluate
)

from optuna import create_study
from optuna.trial import Trial
from optuna.samplers import GPSampler
from itertools import chain

import os
import requests
import pandas as pd
import session_info
import json
import numpy as np

c:\Users\victorm\Documents\nichenetpy\.hatch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
network_path = os.path.normpath("./tutorial_files/model_construction/human")
if not os.path.exists(network_path):
    os.makedirs(network_path)
for filename in (
    "gr_human.csv",
    "lr_network_human.csv",
    "lr_sig_human.csv"
):
    file_path = os.path.join(network_path, filename)
    if not os.path.exists(file_path):
        res = requests.get(f"https://zenodo.org/records/14929618/files/{filename}")
        with open(file_path, "wb") as file:
            file.write(res.content)
train_path = os.path.normpath("./tutorial_files/model_optimization")
if not os.path.exists(train_path):
    os.makedirs(train_path)
for filename in (
    "settings_training_f1234.json",
    "settings_training_f1235.json",
    "settings_training_f1245.json",
    "settings_training_f1345.json",
    "settings_training_f2345.json"
):
    file_path = os.path.join(train_path, filename)
    if not os.path.exists(file_path):
        res = requests.get(f"https://zenodo.org/records/15799578/files/{filename}")
        with open(file_path, "wb") as file:
            file.write(res.content)

In [3]:
gr_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "gr_human.csv")))
lr_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "lr_network_human.csv")))
sig_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "lr_sig_human.csv")))

In [4]:
with open(os.path.join(train_path, "settings_training_f1234.json"), "rb") as file:
    settings_CV = json.loads(file.read())
settings = settings_CV["settings"]

In [5]:
gr_network = gr_network[
    ~ (
        (gr_network["database"] == "NicheNet_LT") &
        np.array([fr in settings_CV["forbidden_ligands_nichenet"] for fr in gr_network["from"]])
    )
    &
    ~ (
        (gr_network["database"] == "CytoSig") &
        np.array([fr in settings_CV["forbidden_ligands_cytosig"] for fr in gr_network["from"]])
    )
]

In [ ]:
source_names = sorted(set(chain(gr_network["source"], lr_network["source"], sig_network["source"])))

def objective(trial:Trial):
    source_weights = dict(
        (
            source_name,
            trial.suggest_float(
                name=source_name,
                low=0,
                high=1
            )
        ) for source_name in source_names
    )
    lr_sig_hub = trial.suggest_float(
        name="lr_sig_hub",
        low=0,
        high=1
    )
    gr_hub = trial.suggest_float(
        name="gr_hub",
        low=0,
        high=1
    )
    ltf_cutoff = trial.suggest_float(
        name="ltf_cutoff",
        low=0.9,
        high=0.999
    )
    damping_factor = trial.suggest_float(
        name="damping_factor",
        low=0.01,
        high=0.99
    )
    res = construct_and_evaluate(
        source_weights,
        lr_sig_hub,
        gr_hub,
        ltf_cutoff,
        damping_factor,
        lr_network,
        gr_network,
        sig_network,
        settings
    )
    return (res[1], res[2])

study = create_study(
    sampler=GPSampler(),
    directions=["maximize", "maximize"]
)
study.optimize(
    objective,
    n_trials=1,
    n_jobs=1
)

[I 2025-08-22 09:54:34,403] A new study created in memory with name: no-name-36ca1ea2-8e6f-4de1-a558-7901c5573d4a
[I 2025-08-22 09:58:34,584] Trial 0 finished with values: [0.024251349121832657, 0.12290020803401419] and parameters: {'CytoSig_all': 0.3476253700670433, 'CytoSig_signature': 0.7388912502771353, 'HTRIDB': 0.6191400684520735, 'HuRi': 0.3384438300602737, 'HuRi_lit': 0.5618156497359045, 'HuRi_union_specific': 0.25800645290873725, 'KnockTF': 0.899971553301746, 'NicheNet_LT_frequent': 0.34940533552292075, 'NicheNet_LT_infrequent': 0.4432239121525092, 'Remap_1': 0.17901417671856956, 'Remap_5': 0.8510503889669311, 'cpdb_complex': 0.08879254773349465, 'cpdb_interaction': 0.8101899565983777, 'evex_binding': 0.7271964750835109, 'evex_catalysis': 0.40537155819622983, 'evex_phosphorylation': 0.3383982506658002, 'evex_regulation_binding': 0.9358458783992891, 'evex_regulation_expression': 0.1023993086785735, 'evex_regulation_other': 0.05032225145518987, 'harmonizome_CHEA': 0.696277495072

the optimization results in a set of pareto optimal solutions

In [7]:
best_params = [trial.params for trial in study.best_trials]
len(best_params)

1

In [8]:
session_info.show()